In [2]:
import pandas as pd
import numpy as np

# Load your file
df = pd.read_excel("/content/HOLY GRAIL MASS INFO DOC.xlsx")

# Check the columns first
print(df.columns.tolist())
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/HOLY GRAIL MASS INFO DOC.xlsx'

In [ ]:
# Make sure key regression columns are numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["FIPS"] = pd.to_numeric(df["FIPS"], errors="coerce")
df["joined"] = pd.to_numeric(df["joined"], errors="coerce")
df["pct_hh65plus_cost_30plus_pct"] = pd.to_numeric(df["pct_hh65plus_cost_30plus_pct"], errors="coerce")

# Keep only rows needed for the first regression
reg_df = df.dropna(subset=[
    "Year",
    "FIPS",
    "joined",
    "pct_hh65plus_cost_30plus_pct"
]).copy()

# Quick check
print(reg_df[["Year", "FIPS", "joined", "pct_hh65plus_cost_30plus_pct"]].head())
print(reg_df.shape)
print(reg_df.dtypes[["Year", "FIPS", "joined", "pct_hh65plus_cost_30plus_pct"]])

In [ ]:

import statsmodels.api as sm

# Define Y (dependent variable)
y = reg_df["pct_hh65plus_cost_30plus_pct"]

# Define X (independent variable)
X = reg_df[["joined"]]

# Add constant (intercept)
X = sm.add_constant(X)

# Run regression
model = sm.OLS(y, X).fit()

# Print results
print(model.summary())

In [ ]:
import statsmodels.formula.api as smf

# Run regression with YEAR fixed effects
model_year_fe = smf.ols(
    "pct_hh65plus_cost_30plus_pct ~ joined + C(Year)",
    data=reg_df
).fit()

print(model_year_fe.summary())

In [ ]:
# County + Year Fixed Effects (THIS is your DiD model)
model_full_fe = smf.ols(
    "pct_hh65plus_cost_30plus_pct ~ joined + C(Year) + C(FIPS)",
    data=reg_df
).fit()

print(model_full_fe.summary())

In [ ]:
#Clustered standard errors by county
model_clustered = smf.ols(
    "pct_hh65plus_cost_30plus_pct ~ joined + C(Year) + C(FIPS)",
    data=reg_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['FIPS']}
)

print(model_clustered.summary())

In [ ]:
reg_df = reg_df.rename(columns={"65plus_poverty_rate": "poverty_65plus_rate"})

In [ ]:
reg_pov65 = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "poverty_65plus_rate",
    "Year",
    "FIPS"
]).copy()

In [ ]:
# ONE CONTROL AT A TIME: 65 plus poverty rate
model_pov65 = smf.ols(
    """
    pct_hh65plus_cost_30plus_pct ~
    joined + poverty_65plus_rate + C(Year) + C(FIPS)
    """,
    data=reg_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['FIPS']}
)

print(model_pov65.summary())

# Switching to PanelOLS package and trying to make things quicker to test out the control variables

In [ ]:
!pip install linearmodels

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

In [ ]:
reg_df = reg_df.rename(columns={"65plus_poverty_rate": "poverty_65plus_rate"})

In [ ]:
panel_pov65 = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "poverty_65plus_rate",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_pov65 = panel_pov65.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
model_pov65 = PanelOLS(
    dependent=panel_pov65["pct_hh65plus_cost_30plus_pct"],
    exog=panel_pov65[["joined", "poverty_65plus_rate"]],
    entity_effects=True,
    time_effects=True
)

result_pov65 = model_pov65.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_pov65.summary)

# Next step is control variable on median household income

In [ ]:
panel_income = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "median_household_income",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_income = panel_income.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
model_income = PanelOLS(
    dependent=panel_income["pct_hh65plus_cost_30plus_pct"],
    exog=panel_income[["joined", "median_household_income"]],
    entity_effects=True,
    time_effects=True
)

result_income = model_income.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_income.summary)

# Control for median home value

In [ ]:
panel_home = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "median_home_value",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_home = panel_home.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
model_home = PanelOLS(
    dependent=panel_home["pct_hh65plus_cost_30plus_pct"],
    exog=panel_home[["joined", "median_home_value"]],
    entity_effects=True,
    time_effects=True
)

result_home = model_home.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_home.summary)

# Control for percent renter

In [ ]:
panel_renter = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "percent_renter",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_renter = panel_renter.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
model_renter = PanelOLS(
    dependent=panel_renter["pct_hh65plus_cost_30plus_pct"],
    exog=panel_renter[["joined", "percent_renter"]],
    entity_effects=True,
    time_effects=True
)

result_renter = model_renter.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_renter.summary)

# Control for education(bachelor's degree)

In [ ]:
panel_edu = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "pct_bachelors_or_higher",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_edu = panel_edu.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
from linearmodels.panel import PanelOLS

model_edu = PanelOLS(
    dependent=panel_edu["pct_hh65plus_cost_30plus_pct"],
    exog=panel_edu[["joined", "pct_bachelors_or_higher"]],
    entity_effects=True,
    time_effects=True
)

result_edu = model_edu.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_edu.summary)

# Controlling percent 65 plus

In [ ]:
panel_65 = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "pct_65plus",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_65 = panel_65.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
from linearmodels.panel import PanelOLS

model_65 = PanelOLS(
    dependent=panel_65["pct_hh65plus_cost_30plus_pct"],
    exog=panel_65[["joined", "pct_65plus"]],
    entity_effects=True,
    time_effects=True
)

result_65 = model_65.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_65.summary)

# Controlling for race

In [ ]:
panel_race = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "pct_black",
    "pct_hispanic",
    "pct_asian",
    "pct_multiracial",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_race = panel_race.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
model_race = PanelOLS(
    dependent=panel_race["pct_hh65plus_cost_30plus_pct"],
    exog=panel_race[[
        "joined",
        "pct_black",
        "pct_hispanic",
        "pct_asian",
        "pct_multiracial"
    ]],
    entity_effects=True,
    time_effects=True
)

result_race = model_race.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_race.summary)

# FINAL MODEL WITH ALL CONTROLS

In [ ]:
panel_final = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "poverty_65plus_rate",
    "median_household_income",
    "median_home_value",
    "percent_renter",
    "pct_65plus",
    "pop_total",
    "FIPS",
    "Year"
]).copy()

In [ ]:
panel_final = panel_final.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
from linearmodels.panel import PanelOLS

model_final = PanelOLS(
    dependent=panel_final["pct_hh65plus_cost_30plus_pct"],
    exog=panel_final[[
        "joined",
        "poverty_65plus_rate",
        "median_household_income",
        "median_home_value",
        "percent_renter",
        "pct_65plus",
        "pop_total"
    ]],
    entity_effects=True,
    time_effects=True
)

result_final = model_final.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_final.summary)

In [ ]:
panel_base = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined",
    "FIPS",
    "Year"
]).copy()

panel_base = panel_base.set_index(["FIPS", "Year"]).sort_index()

from linearmodels.panel import PanelOLS

model_base = PanelOLS(
    dependent=panel_base["pct_hh65plus_cost_30plus_pct"],
    exog=panel_base[["joined"]],
    entity_effects=True,
    time_effects=True
)

result_base = model_base.fit(
    cov_type="clustered",
    cluster_entity=True
)

In [ ]:
import pandas as pd

results_table = pd.DataFrame({
    "Model": [
        "Baseline FE",
        "+ 65+ Poverty",
        "+ Income",
        "+ Home Value",
        "+ Renter Share",
        "+ % Age 65+",
        "Final Model"
    ],
    "Joined Coef": [
        result_base.params["joined"],
        result_pov65.params["joined"],
        result_income.params["joined"],
        result_home.params["joined"],
        result_renter.params["joined"],
        result_65.params["joined"],
        result_final.params["joined"]
    ],
    "Joined SE": [
        result_base.std_errors["joined"],
        result_pov65.std_errors["joined"],
        result_income.std_errors["joined"],
        result_home.std_errors["joined"],
        result_renter.std_errors["joined"],
        result_65.std_errors["joined"],
        result_final.std_errors["joined"]
    ],
    "Joined P-value": [
        result_base.pvalues["joined"],
        result_pov65.pvalues["joined"],
        result_income.pvalues["joined"],
        result_home.pvalues["joined"],
        result_renter.pvalues["joined"],
        result_65.pvalues["joined"],
        result_final.pvalues["joined"]
    ],
    "N": [
        result_base.nobs,
        result_pov65.nobs,
        result_income.nobs,
        result_home.nobs,
        result_renter.nobs,
        result_65.nobs,
        result_final.nobs
    ]
})

# Make it pretty
results_table["Joined Coef"] = results_table["Joined Coef"].round(4)
results_table["Joined SE"] = results_table["Joined SE"].round(4)
results_table["Joined P-value"] = results_table["Joined P-value"].round(4)

print(results_table.to_string(index=False))

In [ ]:
# Make it pretty
results_table["Joined Coef"] = results_table["Joined Coef"].round(4)
results_table["Joined SE"] = results_table["Joined SE"].round(4)
results_table["Joined P-value"] = results_table["Joined P-value"].round(4)

print(results_table.to_string(index=False))

In [ ]:
panel_simple = reg_df.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "joined"
]).copy()

panel_simple = panel_simple.set_index(["FIPS", "Year"])

model_simple = PanelOLS(
    panel_simple["pct_hh65plus_cost_30plus_pct"],
    panel_simple[["joined"]],
    entity_effects=False,
    time_effects=False
)

result_simple = model_simple.fit()

In [ ]:
model_year = PanelOLS(
    panel_base["pct_hh65plus_cost_30plus_pct"],
    panel_base[["joined"]],
    entity_effects=False,
    time_effects=True
)

result_year = model_year.fit(
    cov_type="clustered",
    cluster_entity=True
)

In [ ]:
results_table = pd.DataFrame({
    "Model": [
        "No FE",
        "Year FE",
        "County + Year FE",
        "+ 65+ Poverty",
        "+ Income",
        "+ Home Value",
        "Final Model"
    ],
    "Joined Coef": [
        result_simple.params["joined"],
        result_year.params["joined"],
        result_base.params["joined"],
        result_pov65.params["joined"],
        result_income.params["joined"],
        result_home.params["joined"],
        result_final.params["joined"]
    ],
    "P-value": [
        result_simple.pvalues["joined"],
        result_year.pvalues["joined"],
        result_base.pvalues["joined"],
        result_pov65.pvalues["joined"],
        result_income.pvalues["joined"],
        result_home.pvalues["joined"],
        result_final.pvalues["joined"]
    ]
})

print(results_table.round(4).to_string(index=False))

# EVENT STUDY

In [ ]:
panel_event = reg_df.copy()

In [ ]:
panel_event["event_time"] = panel_event["years_since_joined"]

# Cap extreme values into bins
panel_event.loc[panel_event["event_time"] < -5, "event_time"] = -5
panel_event.loc[panel_event["event_time"] > 5, "event_time"] = 5

In [ ]:
panel_event = panel_event.dropna(subset=[
    "pct_hh65plus_cost_30plus_pct",
    "FIPS",
    "Year"
]).copy()

In [ ]:
event_dummies = pd.get_dummies(panel_event["event_time"], prefix="event")

In [ ]:
print(event_dummies.columns.tolist())

In [ ]:
event_dummies = event_dummies.drop(columns=["event_-1.0"], errors="ignore")

In [ ]:
panel_event = pd.concat([panel_event, event_dummies], axis=1)

event_vars = event_dummies.columns.tolist()
panel_event[event_vars] = panel_event[event_vars].fillna(0)

In [ ]:
panel_event = panel_event.set_index(["FIPS", "Year"]).sort_index()

In [ ]:
from linearmodels.panel import PanelOLS

model_event = PanelOLS(
    dependent=panel_event["pct_hh65plus_cost_30plus_pct"],
    exog=panel_event[event_vars],
    entity_effects=True,
    time_effects=True
)

result_event = model_event.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(result_event.summary)

In [ ]:
import pandas as pd

# Get coefficients and standard errors
event_coefs = result_event.params
event_se = result_event.std_errors

# Convert to DataFrame
event_df = pd.DataFrame({
    "coef": event_coefs,
    "se": event_se
})

# Keep only event variables
event_df = event_df.loc[event_df.index.str.contains("event")].copy()

# Extract event time (the number)
event_df["time"] = event_df.index.str.replace("event_", "").astype(float)

# Sort by time
event_df = event_df.sort_values("time")

print(event_df)

In [ ]:
event_df["ci_lower"] = event_df["coef"] - 1.96 * event_df["se"]
event_df["ci_upper"] = event_df["coef"] + 1.96 * event_df["se"]

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

# Coefficient line
plt.plot(
    event_df["time"],
    event_df["coef"],
    marker='o',
    label="Estimated Effect"
)

# Confidence interval lines
plt.plot(
    event_df["time"],
    event_df["ci_lower"],
    linestyle='--',
    label="95% CI Lower"
)

plt.plot(
    event_df["time"],
    event_df["ci_upper"],
    linestyle='--',
    label="95% CI Upper"
)

# Reference lines
plt.axhline(0, linestyle='-', label="No Effect")
plt.axvline(-1, linestyle=':', label="Reference Period (-1)")

# Axis labels
plt.xlabel("Years Since Joining AARP")
plt.ylabel("Effect on Elderly Housing Cost Burden")

# Title
plt.title("Impact of AARP Age-Friendly Network Over 10 years")

# Legend (THIS is what you wanted!)
plt.legend()

plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.show()